# Actividad de Laboratorio 2
El objetivo de esta Actividad de Laboratorio es implementar y analizar algoritmos simples de clasificación 
en 8 tipos de organismos microscópicos (amebas, euglenas, hidras, paramecios, bacterias (varas/esferas/espiral) 
y  levadura).  Para  ello,  debe  descargar  los  archivos  cell_gal.zip  y  cell_test.zip  que  contienen  las  imágenes 
necesarias para construir los conjuntos de galería y prueba respectivamente.

## P1
Implemente el algoritmo de análisis de textura LBP usando una matriz de 3x3. Aplique LBP a todas las 
imágenes de organismos microscópicos y guarde los resultados. ¿Qué características tienen las imágenes 
procesadas con LBP?   

Respuesta:
Las imágenes LBP resaltan patrones locales de textura, mostrando bordes y detalles finos. Son útiles para distinguir estructuras microscópicas por sus texturas.



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from random import choice

def lbp_3x3(img_gray):
    # Asume que img_gray es una imagen en escala de grises (numpy array)
    lbp_img = np.zeros_like(img_gray)
    for i in range(1, img_gray.shape[0] - 1):
        for j in range(1, img_gray.shape[1] - 1):
            center = img_gray[i, j]
            binary_str = ''
            # Recorrer vecinos en 3x3
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx == 0 and dy == 0:
                        continue
                    neighbor = img_gray[i + dx, j + dy]
                    binary_str += '1' if neighbor >= center else '0'
            lbp_img[i, j] = int(binary_str, 2)
    return lbp_img


input_folder = '02'
output_folder = 'lbp_results'
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.lower().endswith('.jpg'):
        img_path = os.path.join(input_folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        lbp_img = lbp_3x3(img)
        out_path = os.path.join(output_folder, f'LBP_{filename}')
        cv2.imwrite(out_path, lbp_img)

print("Procesamiento LBP terminado. Imágenes guardadas en lbp_results.")



# Ej:
path = os.path.join(input_folder,'cell_gal', 'I1_gal.jpg')
img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
lbp_result = lbp_3x3(img)

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.title('Original')
plt.imshow(img, cmap='gray')
plt.subplot(1,2,2)
plt.title('LBP')
plt.imshow(lbp_result, cmap='gray')
plt.show()

# P2
Desarrolle  un  algoritmo  para  extraer  un  vector  de  características  para  cada  imagen.  Para  ello  divida  la imagen de entrada en 25 regiones de 100x100 pixeles (ancho x alto) y a cada una de ellas calcúlele un 
histograma  cuantizado  en  87  puntos.  Concatene  los  25  histogramas  conseguidos  en  un  vector  de 
dimensiones 1x2175 que será el elemento de salida al método de extracción de características. Adjunte 
este algoritmo a su informe

In [ ]:
#Para ese caso lo que haremos es separar la imagen en 25 regiones de 100x100 pixeles y a cada una de ellas le calcularemos un histograma cuantizado en 87 puntos. 
#Luego concatenamos los 25 histogramas en un vector de tamaño 1x2175.
#Esto entrega las caracteristivas de las regiones las cuales luego concatenamos para obtener el vector final.

def extraer_vector_caracteristicas(img_gray):
    # Asume que img_gray tiene al menos 500x500 píxeles
    regiones = []
    for i in range(5):
        for j in range(5):
            x_ini, x_fin = j*100, (j+1)*100
            y_ini, y_fin = i*100, (i+1)*100
            region = img_gray[y_ini:y_fin, x_ini:x_fin]
            hist, _ = np.histogram(region, bins=87, range=(0,256))
            regiones.append(hist)
    vector = np.concatenate(regiones)
    return vector  # Vector de tamaño 1x2175

# Ejemplo de uso:
img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
vector = extraer_vector_caracteristicas(img)

# P3 
Implemente la siguiente medida de distancia entre dos vectores de largo n: 
$$
d(x,y) = \sum^n_{i=1} |x_i - y_i|
$$


In [ ]:
def medida_distancia(x, y):
    """
    Calcula la distancia Manhattan (L1) entre dos vectores x e y de igual largo.
    """
    return np.sum(np.abs(x - y))   

# P4
Construya  una  base  de  datos  usando  las  imágenes  del  archivo  cell_gal.zip.  Esto  incluye  procesar  las imágenes usando LBP, la posterior extracción y almacenamiento del vector de características para cada 
una de ellas.

In [ ]:
#ocuparemos pandass para manejar la base de datos
import pandas as pd

db = []
input_train = os.path.join(input_folder,'cell_gal')

for filename in os.listdir(input_train):
    if filename.lower().endswith('.jpg'):
        img_path = os.path.join(input_train, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        lbp_img = lbp_3x3(img)
        vector = extraer_vector_caracteristicas(lbp_img)
        db.append({'filename': filename, 'features': vector})

# Guardar la base de datos en un archivo CSV
df_train = pd.DataFrame([{'filename': d['filename'], **{f'f{i+1}': d['features'][i] for i in range(len(d['features']))}} for d in db])

print("Base de datos creada y guardada como base_datos_lbp.csv")


# P5

Realice una prueba de reconocimiento de cada imagen del archivo `cell_test.zip`. Para ello debe procesar cada imagen mediante LBP, extraer el vector de características y compararlo con cada uno de los almacenados en la base de datos usando como medida de similitud la distancia programada. ¿Cuántos reconocimientos correctos se obtienen?

In [ ]:
db = []
input_test = os.path.join(input_folder,'cell_test')

for filename in os.listdir(input_test):
    if filename.lower().endswith('.jpg'):
        img_path = os.path.join(input_test, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        lbp_img = lbp_3x3(img)
        vector = extraer_vector_caracteristicas(lbp_img)
        db.append({'filename': filename, 'features': vector})

labeled_df = pd.DataFrame([{'filename': d['filename'], **{f'f{i+1}': d['features'][i] for i in range(len(d['features']))}} for d in db])

# Calcular las distancias de los vectores
distance_dict = {}

for i, train_row in df_train.iterrows():
    train_label = train_row.iloc[0]
    train_vector = np.array(train_row.iloc[1:])

    if train_label not in distance_dict:
        distance_dict[train_label] = {}

    for j, test_row in labeled_df.iterrows():
        test_label = test_row.iloc[0]
        test_vector = np.array(test_row.iloc[1:])

        dist = medida_distancia(train_vector, test_vector)

        distance_dict[train_label][test_label] = dist

df_classification = pd.DataFrame(distance_dict)
df_classification.to_csv('distances.csv')



In [ ]:

def classifier(df_train, df_test, thresh=100_000, return_list=False):
    """
    Clasifica test images comparando con train images mediante medida de distancia.

    Args:
        df_train (pd.DataFrame): DataFrame con 'filename' en col[0] y features en el resto.
        df_test (pd.DataFrame): DataFrame con 'filename' en col[0] y features en el resto.
        thresh (float): Umbral de distancia para considerar una clasificación válida.
        return_list (bool): Si True, devuelve todas las clases bajo el umbral.
                            Si False, devuelve una sola clase (aleatoria si hay varias).
    Returns:
        dict: {test_filename: predicción o lista de predicciones}
    """
    # 1. Calcular distancias
    distance_dict = {}
    for _, train_row in df_train.iterrows():
        train_label = train_row.iloc[0]
        train_vector = np.array(train_row.iloc[1:])

        if train_label not in distance_dict:
            distance_dict[train_label] = {}

        for _, test_row in df_test.iterrows():
            test_label = test_row.iloc[0]
            test_vector = np.array(test_row.iloc[1:])
            dist = medida_distancia(train_vector, test_vector)
            distance_dict[train_label][test_label] = dist

    # 2. Clasificación usando el umbral
    predicciones = {}
    for test_label in df_test['filename']:
        distancias = {train_label: distance_dict[train_label][test_label] 
                      for train_label in distance_dict}
        
        candidatos = [train_label for train_label, dist in distancias.items() if dist < thresh]
        
        if not candidatos:
            predicciones[test_label] = 'Unknown'
        else:
            if return_list:
                predicciones[test_label] = candidatos
            else:
                predicciones[test_label] = choice(candidatos)

    return predicciones

# Una predicción aleatoria (si hay varias)
preds_unica = classifier(df_train, labeled_df, thresh=120_000, return_list=False)
print(preds_unica)


# P6

¿Qué pasaría si se tiene como entrada al sistema una imagen que no está en la base de datos? ¿Qué estrategia implementaría para enfrentar dicho problema? Pruebe su estrategia usando las imágenes del archivo `cell_impostor.zip`. Identifique al falso impostor, es decir, el organismo que está en la carpeta `cell_impostor` y en la carpeta `cell_gal`, explique el criterio utilizado para detectarlo.


In [ ]:
db = []
input_impostor = os.path.join(input_folder,'cell_impostor')

for filename in os.listdir(input_impostor):
    if filename.lower().endswith('.jpg'):
        img_path = os.path.join(input_impostor, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        lbp_img = lbp_3x3(img)
        vector = extraer_vector_caracteristicas(lbp_img)
        db.append({'filename': filename, 'features': vector})

df_impostor = pd.DataFrame([{'filename': d['filename'], **{f'f{i+1}': d['features'][i] for i in range(len(d['features']))}} for d in db])
preds = classifier(df_train, df_impostor, thresh=120_000)
print(preds)